In [1]:
import pandas as pd
import numpy as np
import json
import ast
import os

In [2]:
PREPATH = '../Data/PreProcessed/'
POSPATH = '../Data/PosProcessed/'
CLASSPATH = '../Data/Classification/'
UDERREVIEWPATH = '../Data/UndeRreview/'

---

#### **01 ) - Split's**

---

---

##### A ) Split para o PosDatasetsClean.csv

In [3]:
df_pos = pd.read_csv(os.path.join(POSPATH,"PosDatasetsClean.csv"))
df_clear = df_pos[df_pos['is_encrypted'] == 0].copy()
df_clear['Columns'] = df_clear['Columns'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
df_input_macro = df_clear[['id', 'Columns']].explode('Columns').rename(columns={'Columns': 'Col'})
df_input_macro = df_input_macro.dropna(subset=['Col'])
df_input_macro.to_csv(os.path.join(UDERREVIEWPATH, 'SplitPosDatasets_UR.csv'), index=False)
df_input_macro.head()

,id,Col
0,CRED-002,status
0,CRED-002,duration
0,CRED-002,credit_history
0,CRED-002,purpose
0,CRED-002,amount


---

##### B ) Instânciando Taxonomia do Artigo

In [4]:
df_article_vars = pd.read_excel(os.path.join(PREPATH, 'PreArticleVars.xlsx'))
df_article_vars['Nome das variáveis'] = df_article_vars['Nome das variáveis'].str.replace('\n', '').str.split(',')
df_article_vars_tidy = df_article_vars.explode('Nome das variáveis')
df_article_vars_tidy['Nome das variáveis'] = df_article_vars_tidy['Nome das variáveis'].str.strip()

df_article_vars_tidy = df_article_vars_tidy[['Nome das variáveis', 'Grupo de Variáveis']]

df_article_vars_tidy.to_csv(os.path.join(POSPATH, 'PosArticleVars.csv'), index=False)
df_article_vars_tidy.head()

,Nome das variáveis,Grupo de Variáveis
0,Home ownership,SOCIOECONOMIC
0,number of children/dependents,SOCIOECONOMIC
0,household size,SOCIOECONOMIC
0,length of employment,SOCIOECONOMIC
0,employment status,SOCIOECONOMIC


---

##### C ) Split para o MacroClassificationTaxonomy.csv

In [5]:
df_standardized_vars = pd.read_csv(os.path.join(CLASSPATH, 'MacroTaxonomyClassification.csv'))
df_standardized_vars.head()

,id,URL,Col,Col_Standardized
0,CRED-002,https://www.kaggle.com/datasets/sid321axn/sout...,status,INSTITUTIONAL and FINANCIAL
1,CRED-002,https://www.kaggle.com/datasets/sid321axn/sout...,duration,INSTITUTIONAL and FINANCIAL
2,CRED-002,https://www.kaggle.com/datasets/sid321axn/sout...,credit_history,INSTITUTIONAL and FINANCIAL
3,CRED-002,https://www.kaggle.com/datasets/sid321axn/sout...,purpose,INSTITUTIONAL and FINANCIAL
4,CRED-002,https://www.kaggle.com/datasets/sid321axn/sout...,amount,INSTITUTIONAL and FINANCIAL


In [6]:
df_split_standardized_vars = df_standardized_vars[df_standardized_vars['Col_Standardized'] != 'UNCLASSIFIED'][['id', 'Col', 'Col_Standardized']].copy()
df_split_standardized_vars.to_csv(os.path.join(UDERREVIEWPATH, 'SplitMacroTC_UR.csv'), index=False)
df_split_standardized_vars.head()

,id,Col,Col_Standardized
0,CRED-002,status,INSTITUTIONAL and FINANCIAL
1,CRED-002,duration,INSTITUTIONAL and FINANCIAL
2,CRED-002,credit_history,INSTITUTIONAL and FINANCIAL
3,CRED-002,purpose,INSTITUTIONAL and FINANCIAL
4,CRED-002,amount,INSTITUTIONAL and FINANCIAL


---

#### **02 ) -**

---

In [7]:
df_MacroTC = pd.read_csv(os.path.join(CLASSPATH, 'MacroTaxonomyClassification.csv'))
df_MacroTC = df_MacroTC[df_MacroTC['Col_Standardized'] != 'UNCLASSIFIED'][['id', 'Col', 'Col_Standardized']]
df_MacroTC.head()

,id,Col,Col_Standardized
0,CRED-002,status,INSTITUTIONAL and FINANCIAL
1,CRED-002,duration,INSTITUTIONAL and FINANCIAL
2,CRED-002,credit_history,INSTITUTIONAL and FINANCIAL
3,CRED-002,purpose,INSTITUTIONAL and FINANCIAL
4,CRED-002,amount,INSTITUTIONAL and FINANCIAL


In [8]:
df_MicroTC = pd.read_csv(os.path.join(CLASSPATH, 'MicroTaxonomyClassificationAdt.csv'))
df_MicroTC = df_MicroTC[['variavel_original', 'conceito_padronizado', 'macro_categoria']]
df_MicroTC.head()

,variavel_original,conceito_padronizado,macro_categoria
0,status,loan_status,INSTITUTIONAL and FINANCIAL
1,duration,payment_term_duration,INSTITUTIONAL and FINANCIAL
2,credit_history,credit_rating,INSTITUTIONAL and FINANCIAL
3,purpose,loan_purpose,INSTITUTIONAL and FINANCIAL
4,amount,loan_amount,INSTITUTIONAL and FINANCIAL


In [9]:
# 1. Extrai os valores únicos de cada coluna como conjuntos
conceitos_macro = set(df_MacroTC['Col'].dropna())
conceitos_micro = set(df_MicroTC['variavel_original'].dropna())

# 2. Subtrai os conjuntos (O que tem no Macro que NÃO tem no Micro?)
conceitos_faltantes = conceitos_macro - conceitos_micro

# 3. Imprime o relatório
print(f"Total de conceitos únicos na tabela Macro: {len(conceitos_macro)}")
print(f"Total de conceitos únicos na tabela Micro: {len(conceitos_micro)}")
print(f"Conceitos do Macro que estão AUSENTES no Micro: {len(conceitos_faltantes)}")

print("-" * 50)

if len(conceitos_faltantes) > 0:
    print("Lista dos conceitos que faltaram ser mapeados no Micro:\n")
    # Imprime a lista em ordem alfabética para facilitar a leitura
    for conceito in sorted(conceitos_faltantes):
        print(f" -> {conceito}")
else:
    print("Sucesso! Todos os conceitos do Macro estão presentes no Micro.")

Total de conceitos únicos na tabela Macro: 594
Total de conceitos únicos na tabela Micro: 591
Conceitos do Macro que estão AUSENTES no Micro: 3
--------------------------------------------------
Lista dos conceitos que faltaram ser mapeados no Micro:

 -> LP_CustomerPayments
 -> MonthlyLoanPayment
 -> Rating Date


In [10]:
df_MacroTC = pd.read_csv(os.path.join(CLASSPATH, 'MacroTaxonomyClassification.csv'))
conceitos_pos = set(df_input_macro['Col'].dropna())
conceitos_macro = set(df_MacroTC['Col'].dropna())

# 2. Subtrai os conjuntos (O que tem no Macro que NÃO tem no Micro?)
conceitos_faltantes = conceitos_pos - conceitos_macro

# 3. Imprime o relatório
print(f"Total de conceitos únicos na tabela Macro: {len(conceitos_pos)}")
print(f"Total de conceitos únicos na tabela Micro: {len(conceitos_macro)}")
print(f"Conceitos do Macro que estão AUSENTES no Micro: {len(conceitos_faltantes)}")

print("-" * 50)

if len(conceitos_faltantes) > 0:
    print("Lista dos conceitos que faltaram ser mapeados no Micro:\n")
    # Imprime a lista em ordem alfabética para facilitar a leitura
    for conceito in sorted(conceitos_faltantes):
        print(f" -> {conceito}")
else:
    print("Sucesso! Todos os conceitos do Macro estão presentes no Micro.")

Total de conceitos únicos na tabela Macro: 789
Total de conceitos únicos na tabela Micro: 786
Conceitos do Macro que estão AUSENTES no Micro: 12
--------------------------------------------------
Lista dos conceitos que faltaram ser mapeados no Micro:

 ->  bank_asset_value
 ->  cibil_score
 ->  commercial_assets_value
 ->  education
 ->  income_annum
 ->  loan_amount
 ->  loan_status
 ->  loan_term
 ->  luxury_assets_value
 ->  no_of_dependents
 ->  residential_assets_value
 ->  self_employed
